## Pré-processamento de dados

Vamos criar um pipeline (uma linha de processamento) que automaticamente extrai os dados de uma planilha csv, gera o corpus de treinamento e teste, gera os iteradores que vão criar os batches (lotes) e codifica as palavras de acordo com algum embedding. 

Nós utilizaremos o corpus de avaliações da B2W, e estaremos interessados em apenas duas colunas: `review_text`, que sera referida daqui para frente como `texto`; e `rating`, que sera referido apenas como `nota`. 

Este pipeline deve conter os seguintes passos:

**filtro** Filtrar todas as linhas cuja nota não é um valor numérico entre 0 e 5.

**partilha** Recebe o nome do arquivo csv contendo o conjunto de dados de entrada, e recebe também o nome do diretório de saída, e as proporções dos conjuntos de dados de treinamento e de teste. A partir dos dados filtrados, altera aleatoriamente a sua ordem e gera as planilhas csv de treinamento e teste. Note que essas planilhas só devem conter as colunas de texto e de nota. As proporções sugeridas são as seguintes: 

• Treinamento: 75%

• Teste: 25%

Se a quantidade de dados for pequena, pode-se aumentar a quantidade de dados de treinamento, com proporções como 85-20, ou 85-15. É usual que o modelo seja salvo após o treinamento. 

**codifica** Utilizar um codificador de palavras para vetor d-dimensional. Pode ser o word2vec re-treinado, mas pode ser também o pacote pré-compilado do Nilc ou uma rede neural do tipo Embedding, a ser treinada com os dados de 

In [16]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras import layers

In [17]:
tf.__version__

'2.21.0'

In [18]:
b2wCorpus = pd.read_csv("data/b2w-10k.csv")
b2wCorpus.head()

,submission_date,reviewer_id,product_id,product_name,product_brand,site_category_lv1,site_category_lv2,review_title,overall_rating,recommend_to_a_friend,review_text,reviewer_birth_year,reviewer_gender,reviewer_state,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18
0,2018-01-01 00:11:28,d0fb1ca69422530334178f5c8624aa7a99da47907c44de...,132532965,Notebook Asus Vivobook Max X541NA-GO472T Intel...,NaN,Informática,Notebook,Bom,4,Yes,Estou contente com a compra entrega rápida o ú...,1958,F,RJ,NaN,NaN,NaN,NaN,NaN
1,2018-01-01 00:13:48,014d6dc5a10aed1ff1e6f349fb2b059a2d3de511c7538a...,22562178,Copo Acrílico Com Canudo 500ml Rocie,NaN,Utilidades Domésticas,"Copos, Taças e Canecas","Preço imbatível, ótima qualidade",4,Yes,"Por apenas R$1994.20,eu consegui comprar esse ...",1996,M,SC,NaN,NaN,NaN,NaN,NaN
2,2018-01-01 00:26:02,44f2c8edd93471926fff601274b8b2b5c4824e386ae4f2...,113022329,Panela de Pressão Elétrica Philips Walita Dail...,philips walita,Eletroportáteis,Panela Elétrica,ATENDE TODAS AS EXPECTATIVA.,4,Yes,SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...,1984,M,SP,NaN,NaN,NaN,NaN,NaN
3,2018-01-01 00:35:54,ce741665c1764ab2d77539e18d0e4f66dde6213c9f0863...,113851581,Betoneira Columbus - Roma Brinquedos,roma jensen,Brinquedos,Veículos de Brinquedo,presente mais que desejado,4,Yes,MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...,1985,F,SP,NaN,NaN,NaN,NaN,NaN
4,2018-01-01 01:00:28,7d7b6b18dda804a897359276cef0ca252f9932bf4b5c8e...,131788803,"Smart TV LED 43"" LG 43UJ6525 Ultra HD 4K com C...",lg,TV e Home Theater,TV,"Sem duvidas, excelente",5,Yes,"A entrega foi no prazo, as americanas estão de...",1994,M,MG,NaN,NaN,NaN,NaN,NaN


In [19]:
import os

In [21]:
output_dir = "data/splits"
train_frac = 0.75
test_frac = 1 - train_frac
random_state = 42

# filtra ratings numéricos entre 0 e 5 e linhas sem review_text
df = b2wCorpus.copy()
# detect which column holds the ratings and convert to numeric
if "rating" in df.columns:
    src_col = "rating"
elif "overall_rating" in df.columns:
    src_col = "overall_rating"
else:
    raise KeyError("No rating column found (expected 'rating' or 'overall_rating')")
df["rating_num"] = pd.to_numeric(df[src_col], errors="coerce")
df = df.dropna(subset=["review_text", "rating_num"])
df = df[(df["rating_num"] >= 0) & (df["rating_num"] <= 5)]

df = df[["review_text", "rating_num"]].rename(columns={"rating_num": "rating"})
df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

# partilha em treino/teste (75/25) mantendo distribuição de classes
train_df, test_df = train_test_split(df, test_size=test_frac, random_state=random_state, stratify=df["rating"])

# salva csvs (somente review_text e rating)
os.makedirs(output_dir, exist_ok=True)
train_path = os.path.join(output_dir, "train.csv")
test_path = os.path.join(output_dir, "test.csv")
train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

# atualiza variáveis usadas adiante
data = df
x = data["review_text"].astype(str).values
y = data["rating"].values
x_train = train_df["review_text"].astype(str).values
y_train = train_df["rating"].values
x_val = test_df["review_text"].astype(str).values
y_val = test_df["rating"].values

print(f"Salvos: {train_path} ({len(train_df)}) e {test_path} ({len(test_df)})")



Salvos: data/splits/train.csv (7499) e data/splits/test.csv (2500)


In [22]:
# parâmetros do codificador/embedding
max_tokens = 20000
max_seq_len = 200
embedding_dim = 100

# camada de tokenização/inteiros
vectorizer = TextVectorization(
    max_tokens=max_tokens, 
    output_mode="int", 
    output_sequence_length=max_seq_len
)
vectorizer.adapt(x_train)
x_train_vec = vectorizer(x_train)
x_val_vec = vectorizer(x_val)
vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)

In [32]:
model_lstm_uni = tf.keras.Sequential([    

    ############ Seu código aqui##################
    # --- EMBEDDING ---
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
    ),

    # --- LSTM ---
    layers.LSTM(128, return_sequences=True),

    ##############################################
    # Conv1D + global max pooling
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.GlobalMaxPooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax'),
])


In [33]:
model_lstm_uni.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=['accuracy'])
model_lstm_uni.fit(x_train_vec, y_train, epochs=5, validation_data=(x_val_vec, y_val))
# model_lstm_uni.evaluate(x=x_val_vec, y=y_val)

Epoch 1/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 44s 175ms/step - accuracy: 0.4479 - loss: 1.3174 - val_accuracy: 0.5156 - val_loss: 1.1252
Epoch 2/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 71s 300ms/step - accuracy: 0.5337 - loss: 1.0563 - val_accuracy: 0.5316 - val_loss: 1.0714
Epoch 3/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 79s 337ms/step - accuracy: 0.6003 - loss: 0.9011 - val_accuracy: 0.5396 - val_loss: 1.1228
Epoch 4/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 75s 317ms/step - accuracy: 0.6609 - loss: 0.7685 - val_accuracy: 0.5296 - val_loss: 1.1652
Epoch 5/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 73s 311ms/step - accuracy: 0.7214 - loss: 0.6496 - val_accuracy: 0.5288 - val_loss: 1.2980


In [34]:
model_lstm_bi = tf.keras.Sequential([    

    ############ Seu código aqui##################
    # --- EMBEDDING ---
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
    ),

    # --- Bidirectional ---
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)),

    ##############################################
    # Conv1D + global max pooling
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.GlobalMaxPooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax'),
])

In [35]:
model_lstm_bi.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=['accuracy'])
model_lstm_bi.fit(x_train_vec, y_train, epochs=5, validation_data=(x_val_vec, y_val))
# model_lstm_bi.evaluate(x=x_val_vec, y=y_val)

Epoch 1/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 118s 470ms/step - accuracy: 0.4457 - loss: 1.3091 - val_accuracy: 0.5312 - val_loss: 1.0570
Epoch 2/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 108s 459ms/step - accuracy: 0.5669 - loss: 0.9765 - val_accuracy: 0.5604 - val_loss: 1.0449
Epoch 3/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 109s 465ms/step - accuracy: 0.6372 - loss: 0.8217 - val_accuracy: 0.5572 - val_loss: 1.0797
Epoch 4/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 109s 463ms/step - accuracy: 0.6986 - loss: 0.7123 - val_accuracy: 0.5488 - val_loss: 1.2857
Epoch 5/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 108s 461ms/step - accuracy: 0.7630 - loss: 0.5768 - val_accuracy: 0.5224 - val_loss: 1.5618


In [36]:
model_gru_uni = tf.keras.Sequential([    

    ############ Seu código aqui##################
    # --- EMBEDDING ---
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
    ),

    # --- GRU ---
    layers.GRU(128, return_sequences=True),

    ##############################################
    # Conv1D + global max pooling
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.GlobalMaxPooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax'),
])

In [37]:
model_gru_uni.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=['accuracy'])
model_gru_uni.fit(x_train_vec, y_train, epochs=5, validation_data=(x_val_vec, y_val))
# model_gru_uni.evaluate(x=x_val_vec, y=y_val)

Epoch 1/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 67s 260ms/step - accuracy: 0.4715 - loss: 1.2469 - val_accuracy: 0.5284 - val_loss: 1.0593
Epoch 2/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 61s 259ms/step - accuracy: 0.5643 - loss: 0.9819 - val_accuracy: 0.5516 - val_loss: 1.0327
Epoch 3/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 54s 230ms/step - accuracy: 0.6162 - loss: 0.8523 - val_accuracy: 0.5476 - val_loss: 1.0902
Epoch 4/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 34s 145ms/step - accuracy: 0.6918 - loss: 0.7222 - val_accuracy: 0.5400 - val_loss: 1.1938
Epoch 5/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 34s 144ms/step - accuracy: 0.7510 - loss: 0.5993 - val_accuracy: 0.5248 - val_loss: 1.3855


In [38]:
model_gru_bi = tf.keras.Sequential([    

    ############ Seu código aqui##################
    # --- EMBEDDING ---
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
    ),

    # --- Bidirectional ---
    layers.Bidirectional(layers.GRU(128, return_sequences=True)),

    ##############################################
    # Conv1D + global max pooling
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.GlobalMaxPooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax'),
])

In [39]:
model_gru_bi.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=['accuracy'])
model_gru_bi.fit(x_train_vec, y_train, epochs=5, validation_data=(x_val_vec, y_val))
# model_gru_bi.evaluate(x=x_val_vec, y=y_val)

Epoch 1/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 55s 222ms/step - accuracy: 0.4551 - loss: 1.3061 - val_accuracy: 0.5312 - val_loss: 1.0649
Epoch 2/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 52s 219ms/step - accuracy: 0.5546 - loss: 1.0038 - val_accuracy: 0.5480 - val_loss: 1.0364
Epoch 3/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 52s 220ms/step - accuracy: 0.6149 - loss: 0.8644 - val_accuracy: 0.5428 - val_loss: 1.1485
Epoch 4/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 51s 219ms/step - accuracy: 0.6757 - loss: 0.7461 - val_accuracy: 0.5192 - val_loss: 1.2258
Epoch 5/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 52s 220ms/step - accuracy: 0.7330 - loss: 0.6302 - val_accuracy: 0.5116 - val_loss: 1.4182


In [40]:
# Avalia os modelos e salva os resultados em CSV no formato exigido
import pandas as pd
models = [
    ("LSTM uni-directional", model_lstm_uni),
    ("LSTM bi-directional", model_lstm_bi),
    ("GRU uni-directional", model_gru_uni),
    ("GRU bi-directional", model_gru_bi),
]
results = []
for name, model in models:
    eval_res = model.evaluate(x=x_val_vec, y=y_val, verbose=0)
    # Keras returns [loss, accuracy] when metrics=['accuracy']
    acc = float(eval_res[1]) if isinstance(eval_res, (list, tuple)) and len(eval_res) > 1 else float(eval_res)
    results.append((name, acc))

# identifica o melhor (maior acurácia)
best_acc = max(r[1] for r in results) if results else None
rows = []
for name, acc in results:
    melhor = 'S' if acc == best_acc else 'N'
    rows.append({'Método': name, 'Acurácia': round(acc, 4), 'Melhor (S/N)': melhor})

df_res = pd.DataFrame(rows)
csv_path = 'resultados.csv'
df_res.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'Salvo: {csv_path}')
df_res

Salvo: resultados.csv


,Método,Acurácia,Melhor (S/N)
0,LSTM uni-directional,0.5288,S
1,LSTM bi-directional,0.5224,N
2,GRU uni-directional,0.5248,N
3,GRU bi-directional,0.5116,N
